# Drum: seluruh lagu dalam satu panggilan

Jalankan pada **VM Colab yang runtime dan bobot ACE-Step-nya masih tersedia**. Salin sel notebook ini ke sesi aktif jika perlu. Helper sudah disertakan, sehingga tidak perlu ZIP tools. Piano asli dipakai kembali jika SOURCE_PATH masih tersedia; kalau tidak, upload piano asli.

Ini uji baru: satu konteks penuh 10–600 detik, tanpa chunk musik independen atau stitching. Untuk piano 4:03, model menerima 244 detik dengan padding akhir kurang dari satu detik, lalu padding dibuang dari ekspor. Base / LEGO / CUDA FP32 / seed 0 / CFG 7 / 50 langkah / caption drum tetap sama. Bobot/runtime dipakai kembali, tidak dipasang ulang.

Hasil START=90 dilaporkan banyak glitch; START=210 dinilai tanpa glitch. Keduanya memakai setting yang sama. Kualitas lagu penuh belum diketahui dan kebutuhan VRAM bisa berbeda. Jika OOM atau output gagal validasi, proses menyimpan report; tidak mengganti mode menjadi chunk atau mengganti precision. Baseline 30 detik dan hasil bass tetap disimpan.

DRY_RUN=True hanya menyiapkan plan. VM kosong perlu sel setup notebook 12 sampai bobot siap dahulu, tanpa menjalankan generasi bassnya.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid
from google.colab import files
DRY_RUN = False
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "drum_probe.py": {
    "sha256": "1ace27dcf94b2ca7846fa6a3aa31114c9a768848a4c2383fd526c513707357dc",
    "gzip_base64": "H4sIAAAAAAAC/7VZW2/bOBZ+969Q80JpVpadBJ1indVDZpouip2dFr3MAhsEBCPRNseypBUlJ2nR/77fIambL21nB+uHxKIOD8/9fIc+Ozt7WTVbr6yKe6m9ovKWTZZNdZGvPK3yVSanpdDaa+jBq9fSK1Wey9S7/vlm+r6Wpffzx5fXXtXktdrK6OzsbKK2ZVHVnqhWpai0bJ9/B8/JsipwlqjXmbr33Iu3eGyJdHMPSRKpdbvySZVLlclJ+ywzmdSVSjjJyY2cQjtRJ5O3727e33zwYu/zxMOHqZQtmEiKRtcqmabQVE935yxk2yKVmXknNbTA4vPpvdASr1K5U4nEu6RJBQsto7KSidKqyLG+zApRX16AVOW6Bk+Z11g23LGopUw1W9zO79zeVaNSkSeS60RkYPwimtPWpaykWcX5oH+OxWJJvFNeFzwpG7b4UDXSMUlEWdvjX+siEzVc0Orl0cneRsHmSVJsS5ErEiny4BdoVwkFh4VeLuqmEpm3qopiJ70HVa+xKdmEns5FJT2Rp16SFRqc12q6FnXoLYssKx56z4u88EpZLYtqSwp58Dlehl65roQJEOKRPuViqxIdeT9nUlRWurWqdejVAsrCcR5cmmlDrUu4wLuX9YOUuWUksZWCEuGYZ08QvPB2BUxnwnMrsyKFzr3pdQQnfZlMJqlcIraSjVhJv2jqsoEGukkomoKFsSIEk7FxFF/CS/cgjhBgzLw0BnHhFv1bla/w3/GZ0caQPbCQ7AujUiTEHe3rt/zlzatfrj/cvAwoGkWVrNVO2jPpA4PBFfAaViG5d2tlMEkXUWIgbMAW+SLTmcpxYru6krmsBHl+Jh9heutZ+7Jjf/AZbtNW2iMM5/M5v+YmbDkF7XwGzZrsO9hTMDaVnLn/LW/LCorslHyItuUlJZp6HK3cLUZ81dJrbdwaKIiU5mRYP1g4S0YPlarlPmHY7Zg4Vs7b/RFf86n1gVOZouBPOPh7nWwsFD2InbWM+bZnEfp8r9o25p1LpY/ErkNdNFUiQ5cAP4S6FlUdz8O0sW6PL+ehfJRJU8v4FZJKutw43BzfUmn2yyCCQYpsB4cYJUvS7shhwZ1hZGtxJOpaJGtDZv1ja3+F2pOb4tGW/LW4eP5jaFTlpkl0xK6rcJnvVFXkFPrtriOv+n2oDFIYy1M0DLbwrFitZNpTbkXCH4pqg7L7MKRDFKDCDQhNXa0kR4/QKM95J76ry4aLrOCb/zQKdMvy8mKwG52Edy2k3frqlzfXHy4v+L/evPvHzbuR2j33ZC1T/ntxH5aZsKZJinypVjEZdsbsg56Ne5wNtzYrLE2fVabuEkVEzUb77j1shs4jH2s/CJ7FtpP2oYkuoqX3m8gaeVNVReUzhxuklrWXqiXamb6yzxUSgBpGkaVeX7XMsQ2YCC+XD27nbIdt1NdsjNhAirabVFU+GRzlPTZdUD4qXfNi42LWhqwkI8WfGWK8btBDXRFFAJ5qzvZUtrDqjWucAQWgtQHMCfe0SIGj/WSak3cJLwBVnO/tzYua9r5BZMCetcobOMT2PnqGWa+okRnAkqybfEPtEskEp9VI9nwVeb9d3yAgn9BVMyogtaxynPoUeb8WHsUd9O2rN+01GAItA00RxLOf3v4TDVwo+J99sS2tS6px0RvWJGtEa9C6euo9DqYiFbWIXUK7BuWPcn7W9S3mKo3521eb9kvQR5I5MGrKlGqWdV3MKLxz4mLaX9we3m9DILeLt84pLW929zdUtXEZdRkcD7LZlizn+UPx25XexqwrlO5/cOyIv8S3bDrdiscp0nSqJdydUuBczvFnOm3R39Siv5C9MMuudU5T0KeyIutVnZPcS+Yq6sAETpDDluGcZA1TPtVr+Lyvvccsc+u+EFBl0wStb1TK/HF5cna77YDoXRD+0HI4X9yNJe1rre9oQlTq+EjVtkKGKHPJxmR6L7DMTincqUBGO6l5YGxqehg3pY/zQSsLyAtW1VNYBwTEyPrGyoklOGnoreD/orne07gPyvhInF71313tHLu9L8Ix5Zl/GPw0CLX4hIUIiVMM2rxFvFNb4i7e4x/ngBhmcuqrZOxGKPDmxj1xG0R2/Lr7CsY0n/E8ZBvBeJqKX4TEKx4V7bHsgyI4AL/sAE73C6Oig8LeBWElkZa5q2CTI9YddFX/0E8HED4YNd3JKM87ypWs/bbDoTG3/OA16gGZzP2e+JbBJxq5+Sw+HwfQYf++wa6E5sjC9KxtmUl6Mi0LXAZWxFN8eAaC5AoDIorXkZA8HC7YoAEQ5I57bONbNiHBnF8hzcgSltpT2qNXngHZBrIbu7hhnexiYo3eg419OUAARGCD21HcMtvHuMWf7O5Z3LeXvVffsqRBQlYSO725QcoDuFFLldiGvRQoQenArEg9sdVhi+fMFpwta6CHFRmYWQrkybGXdh+7G8UhORSAqLstOKja9n6CLebh4b3E8VuJIznKus67ON2TQ4Zxf8Prp5KAUSZXxQiUiWx4v7HH35Il7sbj7y7iDag0XgRSwexOqMjEr3khmlQVLdhaHK3pDKPHPapEThcYmfrkdDCQEuIChhEs6xYAVoEdat5Z064fY9ySkjn0AQPU3FUjVtK9+DKMb5E/+dbPJmg3iNSdGbM24Y4GrdarEarYVqNMLA4D8DqpG5HZ1G1DxGFyNkomB/VtSTH3VDgOiUxJMXx3LwBKuVafTGqd778mvSq0tmJrcpsFlJ5UKI1630qXj3mrUyuxTGfmxE74gdQOCx0trUfvIcZF9Wo4kfmOcGQTm+UO1gbDMlBUaqWAwbtCYAC7Jd9voTO730z0wZ+oJa+pfmBCQMSQfRoaZqyZBv2+l9+BHxgUHYmbFHDVdNbVDMSNgT9IyeEU4O4hekl3SImULxGKklrI95zRXWJ0nLuFb/OlINdVEqa6DlcCoU6XJv7ojuTIjVL0Igj9/piDC6bwPNgzcn+pHAGG+bdsudyWkkbEaV7oOlV0MzMlXpJ8YNbpjxqBckhK8E8sWbhkuyJDEYs/k9hf2NdhDLD1AiCIZeoe8mWwAjG/N2vnf73YjE6BLYK7o5jw+NjUg4GwzZ0jnSR0UwZvk8n9PyW4CRkbsfFevPfeCUIy/XGq1j3BqRMGjcYByGFXP3h7mlEpxYab+xXMykVCpuD3T7UcMzxJxQaw6w9My1d798z7rtqHifIxkWXt/YR+dWO+EiAQ2jMBt/iGjx1wCA1xvGSfqaX65imIOKe7ac6/LLzPZukL+5806vbQHcCedoMrl4EynSILAt3jiyJ7maXM/cXij0gxmaAitzrFMeN8ixTjnFkjlXH7s1J0Xa0MjnhLT5WfSp1UysgTc54WCYY9uyUSacqFo/a7mY6MGNNwiORYCgKkpybFyF1E3Z7fnWJpi/yAKSLkgMgMjv2xrjGlgzxfVUVTxnbntqGenj1xmDtrtNpJbl77A9KDIzr0Zc8xv1R1+l3Ov7a1+92P5jVjRnSOAo0TQEyyU4q3A/SRLeFaZgN/vf/4lsDo+5MmpHujo3LPjxnTjWRfEVbAkOZo2qf97kcCEVmhF11fM5zaaw8RUXz03g9FBMcNnu0Vohtlwx7QtgOGPWg8wptjbZC0k8yijKRt+W340Js2KIZjAiBA7Rv4kzbbUvvdfb8VNWw5W0ndNRwWxxdxZnoycvS/nJKQWGhpuusu0RolgH4p6XaBOvlf2JbzRyweAAA="
  },
  "electric_full_song.py": {
    "sha256": "5629f87436e2168a80fe5e34327d41a12ebc3ffc1ac23d914c54ba21d2c55c25",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAji4UKZVzfQd+oZ9kn6XXWABgpTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ4yqCkaIqpAgKEYg4TaSosjgNIhmJ0zQOZmIp40zm3sHB50IWolxKIb+oosR5mUoSGJdXSalWsEQSCZksFKxxcn0xFA+qXKZVKVRZiEgFiySFaaE4fX/xXXFwPB69hjOFQeaJK5hRpFUeShGmSSm/lEOBJ1mlkYxhSBz/KJIUhsCe4bJK7nHzNBewXBku4cM7cBznQK2yNC9FkC+yIC+k+Q7TbGN+/w5rmN+roFya32lxMM/TlcigLVYzoZuvrSFFNcvyNJRFUbds6p94f16h3GQAJt18A3/H8jJYySILwvpEVaUi8/shWNftX1U2V7E8OPh48ulMTMXrV+Px+OD649nN2Sf4fDwQ8McpwqVcBf5a5oVKE2cijofCURH8cKTG8NEMMOxAM0EQe2D3opTZ0fr4DXZKZ8iLRXKtQokjwioKcEqWy1DplZ05kEL5+hW2q6Qo82olkxI7zAaFlFEBDbfjO73iAi4XJKH0C0AcrvxXb0zT5zKX1A7nwClvsDmd4xaRX6Z+mFXQ+imvpF4JaKPU57gogBZLoLU5YBsuXm5ioENDz3gYAfuWQS6CMExXWZAoPKknTkSUb4YiCcoqD2KRxVV4D8uUSF4ObWP9cZBiRRjLII83QEdArRVvG5RlEN4D0xRLRFSSllLksH+APIF0HwAnzWJkHmAWsQzyVZqosPB6NjkRBZGFWFWFAhjR8WPkGpUADyZpvtKNR/pOuVwAx8G68zSO0wekfuTDTAVJ+l3RswXvD/cGusTReMJokwQrPBKg/ung4CCSc+DKIkyBkPw8TUsX6WIoAP+AqVxG00sA0mDCuHCcc4XXFLNcRQvg06UM75G5DZMjSGZpei8WcToL4gL5EyipAiJdS4AmUEOS4FkMU8MKcBaHD49bA4Ujv9ExBl4uAeVr6Q6ofxXk94B2GOI6ICLmalGMAECyLIiaPeRrJMcyTeNiBLBa+XIdxCMQTV62IUINc5WV0CfLKvO1yMI+3gChARNU5KII0LfGPzmMzwEvcexSlxgBMa3ggKrwkVndAWAlpzbEnz4oL6rmDTSFKhBGAoHarE4r6nvXQwee/AIUHIFYzt0uJPS6uFTveenMgQJwfuQ7nuV5mrtz5+PV1SexCQADkcpUrJZiVt0HSYNJxuxEPOKKT86gCwJspjbAZAQ8XkrEB8BTnwzBAAQaAZ0CICIVlh5KxHu5KVxNW4QrEM7AmgPrzDiTIAHzXF5iaJYaAepINTh20w1guPhVJjIPUBMdfUCiKrRQ2/pzyBM9JE3XQTWyCjIfBZ5/ONKrD4adYSAxfa2NfB7jHzqDDqRVm2os0HhBxK0N/hhMuQyiDQDulm5sX72Z3L4GbGJIz1nLJErz0cnp2dENyvNj783Ig8b1aKaSUbYBZkwcizjbK6Ec2LsUEUOWqgQ4pasyRsS5XhHMZSmTIs0Le6M7Jo1lCuqE6ALkpIxcvi1csrlczRuxTFw9YSB+FscW020TsHMSgfCRM8B4FnSJ1hMXhRJI4f67i8tPwM3JAki7COJgCf8vKwNjNdmWlvaf7wXKCu93gIALNOKCyhuaSw1qBBI76FZQfHgXc3GgQkk8riUswcxnvLgoZTUBcQvACdv2ItaBXte50eKL54GAkA5umxYeSZ7pVDigmHl7xyaFA0tg6Mk1zvYDXH9pgT2TcbUSASAhw//99gGh6Yl/BhAnKEYKGEOytTYHwQoQs3QG++4FuROpWnkUo+MxMB2Ynb5R7v68imMfedZT2SaZDUUcxJUAvQyWpliQCCiU57RwwxfVKCDdvbSBD4abhwRxOwHkAScCll2Dh44GQblgd2s14gC98+rzHK07WwqTEYFmnZdmQOE4m3qBsPKZMyBLG6yFdGLLc5eavIUsaUEQa4Cfoahbk3AJ+hPQ22otglX2oCJQHoOB+AdQjmg4DsUr+K9XJfwGoDPoPfuSAYRBMaFlIVMwNu/ffRXXpx+Of9iW/c059H0H+vrA1mjogAz8Kn1wHxYuy0owPcAOlyVCDwypKdh6UYXCOk2mx/CxCFQyHXt/HYLG/OLXXeCnNBYHnDBWoSrRLVkcaVkM3gfb0AUcPU3AsphtwBoC7fqQour9EcTKIgg3IgbTBxwP2H4DpAokUNbWBhnqQASIZMK3sddz+fdK5ZIaaeQaAUZmB12juQTfoH36mtfcINm4YEODxYyWsLsGzQcrDlAOIh+2u1w4Heg2tLQH9RB0ToBRwdxVQAlrNjLWqCX4SG3BTr4QnE/8JMZmhTF8mJOJn6atk+I3gNoeitfB5uMtqWATDfiAgs/EG/5sI/anqb3HT4hM0ji0NKr28fD4TtNW4015IC/cWxvyrjOfrzK5QN5zjtBrBC8V5fLRGv8v8Sj0yY3KYRZlwhv02wDOEfkrNA4PTkuXuqVGIDYGc7Q0nHUK8k5OH/HwT86uRYNcL4GMx9NDPBR5TEfhhD2qcOUXxz/EUo9lzhjcDVmNTdHlYagAkVZg+0+NVNEjQfDg+oa8glnh6pFHNfBJg+qBezA4d97KMEXvvqaFlSqA2sLlhMgfdX4kHk3vExA6UMgj7/fUFrPcqCUBmK9o2dX8D9o5q3r53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnRa7PcctQZdfGpVcY8ByDdhEsw7XCJxdx29GtG6wT+fWUmP7ig096e7jMYqFjuR4Vki0kE3CCz3Wh2UaowCJkT7yBjYmRgUWBIj5qNAwSnO1UEkQ+8ugQDeJycIwOoMxmafQgyTiYSChcC3JQuzlMyFuixgYNZMsYzpyBZz8Bx98lFzJnT6FM0c/vZyai9LEFGWHo51RSyYBq6PuYk7XrL4OFNg5Mbq9hDEXr3Krl3Xubp165bsB8DmRj23MtdFWL0d4axb/CfEHLa5NbDWhNeN+Rqx64zd7EXtDRtkWtkQ6ZwPhR0IyGmyEYzgiaGfYCQ4LQhxPslJJBS7AC3igOdr3Ag9mabO9R+QwIqxGkRqJdAd2KJ6rkW7ySyjzrDRURvThre4jhZYaOqcFYWrIkU4/vZ+eB2BY83AMD6HhztPQJjRk6oHRxwZGmK5WeAmwLf9ExbatxO72gXFSH+N7OEePtrtrnZUHkfYzyg/sMzrIN2k5gpPeo6Pt9Zbb+o986Ub98RI20QMJHZKKQznEo7Upc/ic5qsVH5EqBt9KJM4domcvaYL+5b1Z3Xa0MFLrS86hqXCLVms61UdhF9LwGpiivPf3Zu9jsA8sAwHcMwrqTzsTD814ms5o4bCGDkI1BM3tDTn3eDSMGsulMUww7Ay09sDhD3o4nlJGdiAMvz3wIIEDg1Xhuq/Ij2GHBiPFl1eXZ0hniAoM8eZATQijRtE9LFHozOI0vBeTaX0iD2MQmqKIgCdmNzIvuCcPHlya2YqyaVjunOD8+9gBwLoG5ofiH/V5EOYKKYJR8aij9r+1gvYa0/D12Ej7SRtvGLpEXod2+vtphyk6k0FJgXjQNU6UPiSthhAILNJf/fMfghzPSmOcE2SRI/Ku7ItQFkYlYKyhFOXY8uhebka0vFjJMoiCMvCcuyemm9p8cxvishc0cVu7bbB7qjYxIzPtsT8XshOU9c2bQxQE/BPMMVhU/zS0hvhsdtQDW1YIE/ZT39Jwl2aqnrQPDIMnze8ajEQ2tQlUr9QyivCcyIlbG2n+3MK1vuXW+A6bYmAhlj6GIWAMM6FDgsNn8odW+uzZwQK/LSVNNsrfQ+kyiXzNS80+uqHvLjZjNPwkI3uPxjw2d0eXDjpAu+0hU5WgUaAJzaCk5f2YRu3/yC8yBNvG5zAEBYwaJ+iQ8iog3cgextyrSaZNOZXWRD0+SkzHosECg3SIRmdVwUCB73jDqVWhVeoRiK0AMyqULRudX79+NdJBEszt5CqyEi18Ckw5cFaRUxTYZOxCCiFyW+1K1adl3wNc+ToBGEEHHB9tTJ6EUSLXzhZSZKr53qca/4VgZ+9Haprc044Nx5tpFWmH+Pb5lewQVBksL4NVv3+pw5dg/69VniZ0CuNmbnfRPHvotG+UfTIrWeQ67879d59/8U9PTt+RcoPvfz371Hx/uH7/y8np384u3+qv06vL84tf3158dCxLPS08vdktLo0xTWtzbjtoD/SqDIPxrj7A1fn5+4vLs6lzDPt8+nhyeXN+9fHD2ccbu0drthYM3T3xazxygKmVVtDVA+zKvET9bgdW++ZrXWpy7GkeLm1jkBo8TONgSLt2GJ6La59+fnsiqqRxMDBtRzFujTlSc6fXn0cfrm/EPIjjWRDeO62z6NyIB/ZaFAPFYkwXJIv+tEfiXLhaYbfBPjiDufpAexWYwz7ndPs7Xsa1lvROQolw0V3WFVH2+JRDX4Fc81mkIiX7TTLHLWQ8BykU5AsQs4eH9w/4azuXxe3MwboygLiXiwMmveC0a020KNE+TyEQ1E47EoEn8SKskgAqZQzqIoPWsCzHYzu316Ya4U7YG71Vn0boUY9+OzmbNkUK87gqlpY/0YliF1VGGdVnwdUFVBvYAB5/HUifbqFBy9DSoS2z3/b1dmMUoNHGfjsCoskIRhmCwvm/8G+LgkyNj+5xuwt0CcfH8IHvv4hAavCZSTvB1Do6G/SUKgbU85ePoGsNpxAim/77T4F/uOO2rhBBice0Z7VtzdJY6Rxs/y26Z8aN6EuXLWzhAgZ0cEAj/yJOQZCgM6DtBBfcoHWqwDEz9Urf2VVSbHxTkdQR20ec0PC0wkHJ4dnJF9h3KxdjND4ZzS0PzzSyUbDQifS0HddomrvDdLCkLzwClofMlVaF+pxZHCSu2VIrBdsGGmL5UHe60VCYU/g9nRkrEbMKht382hyYmvIlZDDi6amhBy6B2uH2AJm2CpB03KddwzTFLBWsMnUKEkF+RhZb13a0oAjAsy6ijcimRRstQGwAo06NmGs063S/ZtWkN9XAbIQXW5j1bXcDRHABDcjBRM0BivWGewpstJjT0D7oAFFDDyho2gBjWFPgtCGBPqTWq3Whj/VjyNVVDjKXgif51HKkuEfjo8w3jbjQ5Gc4jvm8iwYMw2Ql1kTKM/qJCETtjOrNWmuLMMHrKKti6szBjgBOGvKM6dx5JL1AXygs0fzy/SdwH6mpVVmD+pTNQpRIsXX2byQs44ugneFTpGsrIRumFbIcZWCP7TzMizIr/x+JuzoJF4AYXU3RK2SHdPpIh38adrJz+2OU1rp/OCOn5t1cHJhDdIh9fswVU2cEzi/VXSDn5Wupq2eDsKyVEK/OKzoGfSAH7oOF1N7oH0YRuhhtpJOz4dbloX6M5YQJhTqGTdWoD+dcK/ngrbLXiELXWakvVH9hRuJ3a5Clm/8PCKNm8r2p3RZmYzWDg8UAU1ppxq3H//TqvrvmbpyDlAK4omzmX1QtU7ht/01XYWjI1kUqoJKKKi69rypzEII1bNFf3gJ4Gw8taA/23Nbeby5lhLpge0cbUb24s5BH8QRdfuz9m8rOsT6oBhXf1/lCkTqOvGJqrx5/ce2/PTt/D4z3lstbwOxVa9m22+ZUEBqDHFuz83uo4YxmJ4IS7GY9keOszf5m2rD+ZTiljaCJHZ5xbW1dF4gaFTRqxXcsU2e0JV9bt7BHFgyHvkXG47F/Qjj3sUJ6PNJkYYZqpTXSf+sAoCkpIoVSY7YtBhzH+ZuUmQhEsQKNITDUi7qXPFkGhwjmmEy4uvowwqVg/aYWfxUUVD2PpTFhUIFmryND3SzADhprFZShP09WYOE+R0049k8hIhM96cN9t2yyRcJWteJOUmNSp9EaGxqrUkf0uuUNh7sqHIbG0tfWsSbOnkUwVRNkLgitGZbXTXQ5sFX5C3zdN3Hwx6Jfp1cfPlx8uhnuLa/4hvAXayYyWn2tAqwp4IksFjJq7VDH47AKPEJL0KYqHYi3kMUaFsnoMi3PQVdGpppF53tx2hzbwcri6cbMYquXVrKqLY2J23owwVxoVW6biVatN+b/rGIDexCmmXwMSuhiPI6w7jMTWg9yTPAkUvM5lrrzaxIwGAo6JDhwJsvRKCKtYyMOJv7ZhfGTbsG5CdaZbQfbOLKvRxXnJs/aejikHxHoNz8c1k7zzbcl5LXQoyQbmeSYzGE5D3uZVzCU4bHcN9YFGEKe0AshLwKpij9cxtuW+nWMY+Zj+lrhPM1Qt+SDYZqt41FtZVme9bC2d+UAFb6BKhp3Vye6sGwW74Xvp7A8UiVVWlFwrHlHRRm7SGZgRCMrN2+okBrM8ynx28kZVWuUiuKhoD7IHfHEuyCPHgAF4KCURPzm0QpsUJD1Em7wHRZookRicp3r0kpJiPacPRmWHvXMqOzx5KycWH+dWSuoMRSdak27nhV36LpwGKBIaCKZBdN2vudbb2BxjlYFk54Y59y5fn9yOaEHb49m49vv2rmy7+6eCoNTCk//ev1Z1E+qno988rEOujtfXZ6J65Obm4m9c7PnxHs9f9LVysNnDqcj6EyhzXO9msZ2nNGqlEHrfLtcXtdfk0r0Sdj4vqUZyaE44pwbWfxHOMtpcgs8AHDWsv8HTXSQSIXCYyz4ZD73yR3DWGdDN7Uia6pXQA9OdyZ7tt2KhvxuHT6wCX9Q/LJnd1P2pbXnVnjNUkRuX9huy5YdtJSUTaTWSIr+a0nK0X9jAkUOpvnw2UYz/NYB7V1gjRqMPH5x0XlKIgt9eipBt0L7TEgqWachxx6bc8JOGFTc2vt2fGe9JaI3Qv1xzC273LEwgwY6RVGNTVI/RYLfQ0qOtkCmZ5i8KaljMvJ3pE9wBKx020nB3+GImrW6nc8B1E6K6APxzuzgoM2g5oohqR0LC6Acyqb6C9BH5PDiEnAKWZZUDXJ36/Ag1G99vTzVirZLg2JUyM0TTtaqVnhePx7FRPyw79Xorjej/eUrdlFBDctOvQFuWYL/Q4F81JuxXKTtp61B3HqCurUNGNTgZ/n8RlN9NVuSNYKrg6RD98pqAhcLGLr0G0hwT//6ZjTeoehZBV+9VMFCmq4nmyDx1QEji0jwXm6I+ujBAJmL0DLUn2AwGkR5Ch/TuYNnX4ycsG5vvcag/YCH80Ibrj12K9tZI1Pt1GIiJh8tc+gtMZ0ZX1XhozGrd4ZFAH6hvmqmOu4OQBjlIJjTFfE2jNKFewSp5y73OakJ14q34kIj2lnURG8pL/bed0rifi+/XwaThYdqEH8QtugHPQvk2XRJGgY0ewsCF8EHnwY8lA+AW+NbMNCgmHvENKXFmugRt1ZrSXndw0K+NUzz0ZZE675B6ZtEvMZzWulQx9SX0426KyEph1qQ7l0h7LkI2zx4EQIjwxA2Gbz8TVSjmrCchnXSSOceRDvjYADbLk5vC/VuFdfd1uudbnlYHaeyqtIGezXFy5ds1St3Fm1Vk909C7ALUjL4Nm3BT0rQu7O1uZVDHNgqoklP1I96a+2CTioaeiiwm+P3RVKbc7dq0+52bLV/sZ7eVnx212bNY7bnbogx2K3NOjH256/U78I0NtqwFlXTbYWNKTyiYl9T8VT/vSvabOlgnauzrZytXmdn2DqTwT2/uQZeRNsOjKzZBhzF1oI7R71kYZ1m2b9ue9DuZYkMmRGmW7y0TT9g8CMmd01ox/Z37cn0htWP1vtImFnmIN1N25uhIS+mjenzJPOt3msnEXWw26t8YRa1n3b/FxlUHeDeZe93HSBbYuuprRhW2z3jT/LOLAVv5vWq8W+FNEY7dgX+LalCQK4BjLWxIGPFX8RHIAHFz1zqzGLNmyOW/GzRSDPZ+4NZ6BddRz96h7OYiDyVtqA/ZspcvJN8Qbb2NfW4keQIJMbNfT9KQ/DvrZn47yj4gZ7iNv49Usj0mnIKsGUAjD7dFSPQ/+JEcXt8t3fl+t+ZqNfeO5xjCtY5TFjU8vl37USFy3oymTPNLcZ7Z1rPzezJ+I80TZ2rlSo7b+1MlGwVfFGraoVPZYwFu2MHHaqiWhpCCijkFPO4cKn9M+soTM9EfcSaCG4+X6NHeNMqhdHr0l+4ssm5om+DdYm8gV2T0Sq5pjFIHnbihBoBVd1/hkO2fAKzQ1E/H+MXVPpEkq0dQyLYb5DtdBUz3KNOGdUnGtrLN4cyuSPubMcsubGpXjfZJGrWH4NORM+5IYk6qTny1ljG6Pv+h+C6BOw2J0CWVVgHyDKWnAefSyN99KWJkw/+Bzrm61BTTAAA"
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_probe_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
single = importlib.util.module_from_spec(spec); spec.loader.exec_module(single)
sys.modules["electric_full_song"] = single
ROOT = single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = single.discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ditemukan. Gunakan VM aktif dari notebook 12 atau jalankan sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    single.model_python(ROOT)
spec = importlib.util.spec_from_file_location("drum_probe_embedded", TOOLS / "drum_probe.py")
drum = importlib.util.module_from_spec(spec); spec.loader.exec_module(drum)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
print("Runtime dipakai kembali:", ROOT)
print("Piano:", SOURCE_PATH)


## Satu generasi seluruh lagu

START=0 dan duration=None memakai seluruh piano asli (maksimum 600 detik). Tidak ada uji durasi atau chunk tambahan. Folder baru dibuat untuk setiap percobaan. Setelah error, jalankan sel download untuk mengambil report yang tersedia.


In [ ]:
from IPython.display import Audio, display
START = 0
DRUM_OUT = ROOT / "runs" / ("drum_full_" + uuid.uuid4().hex[:10])
report = drum.generate(ROOT, SOURCE_PATH, DRUM_OUT, start=START, duration=None, execute=not DRY_RUN)
print("Status:", report["status"], "|", DRUM_OUT)
if not DRY_RUN:
    for name in ("drums_preview.mp3", "mix_preview.mp3"):
        print(name)
        display(Audio(filename=str(DRUM_OUT / name), normalize=False))


## Simpan hasil penuh dan kirim feedback

Simpan drum_result.zip sebagai arsip WAV drum/mix. Upload drum_feedback.zip ke chat untuk menilai hasil seluruh lagu. Jika generasi gagal, feedback hanya berisi report yang tersedia.


In [ ]:
for name in ("drum_result.zip", "drum_feedback.zip"):
    path = DRUM_OUT / name
    if path.is_file():
        files.download(str(path))
if DRY_RUN:
    print("Plan saja; tidak ada audio model atau ZIP hasil yang dibuat")
